# 09: Expert re-labeling (val, exploratory)
Runs analysis A4 pre-declared in `research/analysis_plan.md` (commit `2ad5626`) on the 266 val images of the expert four-findings "validation" file. Uses the frozen score and thresholds from `research/operating_points.json`; nothing here changes the model, the score or the thresholds. Val results are exploratory; test is confirmatory.

In [1]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.bootstrap import bootstrap_auroc, bootstrap_mean, patient_resample_counts, percentile_ci
from src.data import (EXPERT_FOUR, EXPERT_TO_NIH, NIH_DISEASES, load_dev_expert_four, load_dev_expert_readers,
                      load_dev_labels)
from src.metrics import auroc, fleiss_kappa
from src.operating_points import sens_spec, sha256_file
from src.paths import DEV_OUTPUT_DIR, RESEARCH_DIR, SPLIT_CSV

load_dotenv()
DATA_ROOT = Path(os.getenv("DATA_ROOT"))
NIH_DIR = DATA_ROOT / "NIH_Chest_X-Rays"
EXPERT_DIR = DATA_ROOT / "NIH_expert_labels"
FOUR_CSV = EXPERT_DIR / "nih-cxr-14_labels_four_findings_expert_labels_validation_labels.csv"
READERS_CSV = EXPERT_DIR / "nih-cxr-14_labels_four_findings_expert_labels_individual_readers.csv"

## Load the frozen primary score (hash-checked)

In [2]:
frozen = json.loads((RESEARCH_DIR / "operating_points.json").read_text())
df = load_dev_labels(NIH_DIR, SPLIT_CSV)
val_df = df[df["split"] == "val"].reset_index(drop=True)
labels = val_df[NIH_DISEASES].to_numpy(dtype=float)
LOGIT_COLS = [f"logit_{d}" for d in NIH_DISEASES]

member_logits = {}
for m in frozen["members"]:
    csv = DEV_OUTPUT_DIR / m["run_id"] / "val_logits.csv"
    if sha256_file(csv) != m["val_logits_sha256"]:
        raise RuntimeError(f"{csv.name} of {m['run_id']} doesn't match the hash in operating_points.json")
    saved = pd.read_csv(csv)
    if not saved["Image Index"].equals(val_df["Image Index"]):
        raise RuntimeError(f"{m['run_id']}: saved rows don't line up with val_df")
    member_logits[m["run_id"]] = saved[LOGIT_COLS].to_numpy()

z = np.mean(list(member_logits.values()), axis=0)  # the frozen primary score: mean of member logits
thr = np.array([frozen["thresholds"][d] for d in NIH_DISEASES])
val_sens = [sens_spec(z[:, i], labels[:, i], thr[i])[0] for i in range(len(NIH_DISEASES))]
print(f"Members: {list(member_logits)} (val-logit hashes match)")
print(f"Frozen thresholds on the rebuilt score: val sensitivity min {min(val_sens):.4f}, max {max(val_sens):.4f}")

Members: ['20261109-1844_none_s0', '20261109-1858_none_s1'] (val-logit hashes match)
Frozen thresholds on the rebuilt score: val sensitivity min 0.9001, max 0.9091


## Expert labels (dev only) and evaluability, decided from label counts before any score

In [3]:
expert = load_dev_expert_four(FOUR_CSV, df)
print("Expert four-findings 'validation' file, dev images per split:", expert["split"].value_counts().to_dict())

ex = (val_df.assign(row=np.arange(len(val_df)))
      .merge(expert, on=["Image Index", "split"], how="inner", validate="one_to_one"))
r = ex["row"].to_numpy()  # positions of the expert images inside val_df (and inside z)
print(f"Val: {len(ex)} images, {ex['Patient ID'].nunique()} patients, "
      f"AP {(ex['View Position'] == 'AP').mean():.0%} (all val: {(val_df['View Position'] == 'AP').mean():.0%})")

y_exp, y_nih, rows = {}, {}, []
for f in EXPERT_FOUR:
    y_exp[f] = ex[f"expert {f}"].to_numpy()
    if f in EXPERT_TO_NIH:
        y_nih[f] = ex[EXPERT_TO_NIH[f]].max(axis=1).to_numpy().astype(int)  # NIH ref = any mapped NIH label
    n_nih = int(y_nih[f].sum()) if f in y_nih else None
    rows.append({"finding": f, "expert pos": int(y_exp[f].sum()),
                 "expert pos patients": ex.loc[y_exp[f] == 1, "Patient ID"].nunique(),
                 "NIH pos": n_nih, "evaluable": n_nih is not None and y_exp[f].sum() >= 10 and n_nih >= 10})
counts_table = pd.DataFrame(rows).set_index("finding")
EVALUABLE = counts_table.index[counts_table["evaluable"]].tolist()
print(counts_table)
print("\nEvaluable on val (>= 10 positives under both labels, decided before any score):", EVALUABLE)

Expert four-findings 'validation' file, dev images per split: {'train': 2148, 'val': 266}
Val: 266 images, 103 patients, AP 89% (all val: 32%)
                  expert pos  expert pos patients  NIH pos  evaluable
finding                                                              
Fracture                   2                    2      NaN      False
Pneumothorax               0                    0      0.0      False
Airspace opacity         117                   43     91.0       True
Nodule or mass            35                   18     28.0       True

Evaluable on val (>= 10 positives under both labels, decided before any score): ['Airspace opacity', 'Nodule or mass']


In [4]:
for f in EXPERT_TO_NIH:
    agree = pd.crosstab(pd.Series(y_nih[f], name="NIH"), pd.Series(y_exp[f], name="expert"))
    print(f"\n{f}: label agreement (rows NIH, columns expert)\n{agree}")

nih_no_finding = labels[r].sum(axis=1) == 0
any_expert = ex[[f"expert {f}" for f in EXPERT_FOUR]].to_numpy().any(axis=1)
print(f"\nNIH 'No Finding' images: {nih_no_finding.sum()}, of which {(nih_no_finding & any_expert).sum()} "
      f"have an expert-positive finding")


Airspace opacity: label agreement (rows NIH, columns expert)
expert    0   1
NIH            
0       123  52
1        26  65

Nodule or mass: label agreement (rows NIH, columns expert)
expert    0   1
NIH            
0       221  17
1        10  18

Pneumothorax: label agreement (rows NIH, columns expert)
expert    0
NIH        
0       266

NIH 'No Finding' images: 134, of which 35 have an expert-positive finding


## A4 primary: AUROC against expert vs NIH labels on the same images (estimation)
Score = max of the frozen score over the mapped NIH classes. Patient bootstrap over the 103 patients of this subset (B = 2000, seed 0); both AUROCs and their difference come from the same resamples (paired).

In [5]:
N_BOOT = 2000
counts, codes = patient_resample_counts(ex["Patient ID"], n_boot=N_BOOT, seed=0)  # resample the 103 patients


def finding_score(logits: np.ndarray, f: str) -> np.ndarray:
    cols = [NIH_DISEASES.index(c) for c in EXPERT_TO_NIH[f]]
    return logits[r][:, cols].max(axis=1)  # max over the mapped NIH classes


def with_ci(point: float, boot: np.ndarray, sign: str = "") -> str:
    lo, hi = percentile_ci(boot)
    return f"{point:{sign}.3f} [{lo:{sign}.3f}, {hi:{sign}.3f}]"


boot_exp, rows = {}, []
for f in EVALUABLE:
    s = finding_score(z, f)
    boot_exp[f] = bootstrap_auroc(s, y_exp[f], counts, codes)
    boot_nih = bootstrap_auroc(s, y_nih[f], counts, codes)  # same scores, same resamples -> paired
    a_exp, a_nih = auroc(s, y_exp[f]), auroc(s, y_nih[f])
    rows.append({"finding": f, "AUROC expert": with_ci(a_exp, boot_exp[f]), "AUROC NIH": with_ci(a_nih, boot_nih),
                 "expert minus NIH": with_ci(a_exp - a_nih, boot_exp[f] - boot_nih, "+"),
                 "NaN reps": int(np.isnan(boot_exp[f]).sum() + np.isnan(boot_nih).sum())})
print(pd.DataFrame(rows).set_index("finding").to_string())

                          AUROC expert             AUROC NIH         expert minus NIH  NaN reps
finding                                                                                        
Airspace opacity  0.911 [0.865, 0.943]  0.801 [0.711, 0.878]  +0.110 [+0.031, +0.198]         0
Nodule or mass    0.808 [0.667, 0.905]  0.817 [0.669, 0.928]  -0.009 [-0.139, +0.087]         0


## A4 secondary: sensitivity / specificity at the frozen thresholds; ensemble minus members
A finding is predicted positive if any of its mapped NIH classes is at or above its frozen threshold.

In [6]:
pred = z[r] >= thr  # per NIH class: score at or above its frozen threshold
rows = []
for f in EVALUABLE:
    hit = pred[:, [NIH_DISEASES.index(c) for c in EXPERT_TO_NIH[f]]].any(axis=1)  # any mapped class fires
    for name, y in [("expert", y_exp[f]), ("NIH", y_nih[f])]:
        pos = y == 1
        sens_ci = percentile_ci(bootstrap_mean(hit[pos].astype(float), counts, codes[pos]))
        spec_ci = percentile_ci(bootstrap_mean((~hit[~pos]).astype(float), counts, codes[~pos]))
        rows.append({"finding": f, "labels": name, "pos": int(pos.sum()), "sens": hit[pos].mean(),
                     "sens_lo": sens_ci[0], "sens_hi": sens_ci[1], "spec": (~hit[~pos]).mean(),
                     "spec_lo": spec_ci[0], "spec_hi": spec_ci[1]})
print(pd.DataFrame(rows).set_index(["finding", "labels"]).round(3).to_string())

                         pos   sens  sens_lo  sens_hi   spec  spec_lo  spec_hi
finding          labels                                                       
Airspace opacity expert  117  0.983    0.951    1.000  0.490    0.357    0.609
                 NIH      91  0.967    0.929    1.000  0.411    0.285    0.529
Nodule or mass   expert   35  0.886    0.708    0.977  0.429    0.329    0.524
                 NIH      28  0.893    0.692    1.000  0.420    0.317    0.515


In [7]:
for f in EVALUABLE:
    a_ens = auroc(finding_score(z, f), y_exp[f])
    for run_id, logits_m in member_logits.items():
        s_m = finding_score(logits_m, f)
        lo, hi = percentile_ci(boot_exp[f] - bootstrap_auroc(s_m, y_exp[f], counts, codes))
        print(f"{f}, expert AUROC, ensemble minus {run_id}: {a_ens - auroc(s_m, y_exp[f]):+.4f}  "
              f"paired 95% CI [{lo:+.4f}, {hi:+.4f}]")

Airspace opacity, expert AUROC, ensemble minus 20261109-1844_none_s0: +0.0056  paired 95% CI [-0.0049, +0.0173]
Airspace opacity, expert AUROC, ensemble minus 20261109-1858_none_s1: +0.0005  paired 95% CI [-0.0114, +0.0109]
Nodule or mass, expert AUROC, ensemble minus 20261109-1844_none_s0: +0.0061  paired 95% CI [-0.0174, +0.0416]
Nodule or mass, expert AUROC, ensemble minus 20261109-1858_none_s1: +0.0066  paired 95% CI [-0.0301, +0.0322]


## Readers (descriptive only)
Per finding: distribution of YES reads, HEDGE reads, Fleiss' kappa (YES vs not YES; HEDGE counted as not YES) on images with exactly 3 reads. No model-vs-reader comparison.

In [8]:
readers = load_dev_expert_readers(READERS_CSV, df)
rv = readers[readers["split"] == "val"]
reads = rv.groupby("Image Index").size()
print(f"Val reader rows: {len(rv)} on {rv['Image Index'].nunique()} images, {rv['Reader ID'].nunique()} readers; "
      f"reads per image {reads.value_counts().sort_index().to_dict()}")
if not set(rv["Image Index"]) <= set(ex["Image Index"]):
    raise RuntimeError("A val reader image is missing from the aggregated expert labels")

three = rv[rv["Image Index"].isin(reads.index[reads == 3])]  # Fleiss' kappa needs the same number of reads per image
rows = []
for f in EXPERT_FOUR:
    n_yes = (three[f] == "YES").groupby(three["Image Index"]).sum()  # HEDGE counts as not YES
    table = np.stack([n_yes.to_numpy(), 3 - n_yes.to_numpy()], axis=1)
    dist = n_yes.value_counts().reindex(range(4), fill_value=0)
    rows.append({"finding": f, **{f"{k}/3 YES": int(dist[k]) for k in range(4)},
                 "HEDGE reads": int((rv[f] == "HEDGE").sum()), "Fleiss kappa": fleiss_kappa(table)})
print(f"\nImages with exactly 3 reads: {three['Image Index'].nunique()}")
print(pd.DataFrame(rows).set_index("finding").round(3))

Val reader rows: 795 on 266 images, 13 readers; reads per image {2: 3, 3: 263}

Images with exactly 3 reads: 263
                  0/3 YES  1/3 YES  2/3 YES  3/3 YES  HEDGE reads  \
finding                                                             
Fracture              258        5        0        0            0   
Pneumothorax          261        2        0        0            5   
Airspace opacity      131       32       37       63            0   
Nodule or mass        222       18        8       15           19   

                  Fleiss kappa  
finding                         
Fracture                -0.006  
Pneumothorax            -0.003  
Airspace opacity         0.626  
Nodule or mass           0.634  


## Decisions and observations (val, exploratory; nothing here changes the model, the score or the thresholds)
- Subset: 266 val images, 103 patients, 89% AP (all val 32%); not a random sample of val. Evaluable (>= 10 positives under both labels, from counts before any score): Airspace opacity (117 expert / 91 NIH positives) and Nodule or mass (35 / 28). Fracture (2 expert positives, no NIH class) and Pneumothorax (0) are counts only.
- Label agreement: Airspace opacity 52 expert-positive images are NIH-negative and 26 the reverse; Nodule or mass 17 and 10. 35 of the 134 NIH "No Finding" images have an expert-positive finding.
- A4 primary: Airspace opacity AUROC 0.911 [0.865, 0.943] against expert vs 0.801 [0.711, 0.878] against NIH labels; expert minus NIH +0.110 [+0.031, +0.198]. Nodule or mass 0.808 [0.667, 0.905] vs 0.817 [0.669, 0.928]; -0.009 [-0.139, +0.087] (uninformative). Consistent with NIH label noise attenuating AUROC for Airspace opacity; mapping mismatch and case mix are not ruled out.
- At the frozen thresholds (union of mapped classes): sensitivity 0.983 / 0.886 (expert labels), specificity only 0.490 / 0.429, expected from the union rule and the AP-heavy subset.
- Ensemble minus members (expert AUROC): all paired CIs include 0 (e.g. Airspace opacity +0.0056 [-0.0049, +0.0173] vs s0); inconclusive at this sample size.
- Readers: Fleiss' kappa 0.626 (Airspace opacity) and 0.634 (Nodule or mass) on 263 images with 3 reads. Fracture -0.006 and Pneumothorax -0.003 are not interpretable (near-zero prevalence: chance agreement ~0.99).
- Next: test-time A4/A5 run only through the test entry point after the plan is frozen.